In [ ]:
import numpy as np
import networkx as nx
import re
from itertools import combinations
from collections import Counter
from argparse import Namespace

from qiskit import QuantumCircuit, transpile, generate_preset_pass_manager
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.transpiler import Layout
from qiskit.transpiler.passes import LayoutTransformation
from qiskit.converters import dag_to_circuit, circuit_to_dag



from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper
from qiskit_qaoa.utils.hamiltonian_utils import hamiltonian_to_interactions
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
from qiskit_qaoa.utils.pass_managers import get_hubo_pass_manager

In [ ]:
rng = np.random.default_rng(seed=1)

In [ ]:
args = Namespace(**dict(
    filename='trivial',
    extra=1,
    fraction_four=0.0,
    fraction_six=1.0,
    timeout=10,
    copy_numbers=[1,1,1],
    coupling_map='grid',
    reps=2,
    method='Powell',
    shots=1000,
    init='random'
))

In [ ]:

def two_qubit_count(qc: QuantumCircuit):
    ops: dict[str, int] = qc.count_ops()
    return ops.get("cz", 0) + ops.get("rzz", 0) + ops.get("cx", 0) + ops.get("swap", 0)
   
    
def print_circuit_info(qc: QuantumCircuit, circuit_name: str):
    print(f'{circuit_name} has {two_qubit_count(qc)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')
    
    
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{args.filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, args.copy_numbers)
num_qubits = n*T
print(f'Virtual qubits: {num_qubits}')


if args.coupling_map == 'line':
    extended_swap_strat = ExtendedSwapStrategy.from_line(list(range(num_qubits)), num_swap_layers=1000)
elif args.coupling_map == 'grid':
    extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)
else:
    raise Exception('Invalid coupling map type')

num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map
    
coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)
dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)



print(f'Physical qubits: {num_physical_qubits}')

basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"]

backend_options = dict(
    method='statevector',
    device='GPU',
    precision='single',
    basis_gates=basis_gates
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map)
print(backend.configuration().to_dict()["n_qubits"])

full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, fraction_terms=1.0)

hamiltonians, swap_depths, edge_maps, compiled_circuits = {}, {}, {}, {}
for layer in range(args.reps):
    print(f'Getting hamiltonian for layer {layer}')
    hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, fraction_terms=(layer/args.reps, (layer+1)/args.reps))
    hamiltonians[layer] = hamiltonian
    
    all_pauli_z = np.array(
        [i.paulis[0].z for i in hamiltonian]
    )
    print(f'Hamiltonian: {len(hamiltonian)}')
    print(f'Orders: {Counter(np.sum(all_pauli_z, axis=1))}')
    
    program_interactions = hamiltonian_to_interactions(hamiltonian, args.fraction_four, args.fraction_six)
    lengths = Counter([len(interaction) for interaction in program_interactions])

    print(f'Program interactions: {len(program_interactions)}')
    print(f'Orders: {Counter([len(interaction) for interaction in program_interactions])}')
    
    mapper = HigherOrderSatMapper(timeout=args.timeout)

    best_circuit_depth, best_swap_depth, best_edge_map, best_circuit = np.inf, 0, {x: x for x in range(num_physical_qubits)}, QuantumCircuit(num_physical_qubits)
    depths = sorted(list(set([int(x) for x in np.linspace(0, len(extended_swap_strat._swap_layers), 5)])))
    for depth in depths:
        print('--------------------------------------------------')
        sat_results = mapper.hubo_max_sat(
            num_qubits, program_interactions, extended_swap_strat, depth
        )
        if sat_results is None:
            print('No results')
            continue
        mapping = sat_results[depth][1]
        edge_map = dict(mapping)

        print(f'Cost: {sat_results[depth][0]}')
        print(edge_map)

        pm = get_hubo_pass_manager(extended_swap_strat, depth, args.extra)

        new_cost_circ = QuantumCircuit(num_physical_qubits)
        new_cost_circ.append(PauliEvolutionGate(hamiltonian, time=Parameter("γ")), [edge_map[i] for i in range(num_physical_qubits)])
        new_tcost = pm.run(new_cost_circ)
        
        print_circuit_info(new_tcost, 'Remapped, commuting gate routed circuit')
        print(new_tcost.count_ops())
        
        circuit_depth = new_tcost.depth(lambda instr: len(instr.qubits) > 1)
        if circuit_depth < best_circuit_depth:
            best_circuit_depth = circuit_depth
            best_swap_depth = depth
            best_edge_map = edge_map
            best_circuit = new_tcost
            
    
    swap_depths[layer] = best_swap_depth
    edge_maps[layer] = best_edge_map
    compiled_circuits[layer] = best_circuit

In [ ]:
compiled_circuits[0].draw(fold=-1)

In [ ]:
edge_maps[0]

In [ ]:
edge_maps[1]

In [ ]:
layout = Layout({i: compiled_circuits[0].qubits[edge_maps[0][i]] for i in range(num_physical_qubits)})
layout

In [ ]:
for instruction in compiled_circuits[0].data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            layout.swap(compiled_circuits[0].qubits[int(matches[0])], compiled_circuits[0].qubits[int(matches[1])])
        else:
            raise Exception('Did not find 2 swap indices')
layout

In [ ]:
layout1 = Layout({i: compiled_circuits[1].qubits[edge_map[i]] for i in range(num_physical_qubits)})

permutation = {
    pqubit: layout1.get_virtual_bits()[vqubit]
    for vqubit, pqubit in layout.get_virtual_bits().items()
}
permutation

In [ ]:
transformation_pass = LayoutTransformation(extended_swap_strat._coupling_map, from_layout=layout, to_layout=layout1)

In [ ]:
swap_qc = QuantumCircuit(num_physical_qubits)
swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))

In [ ]:
swap_qc.draw(fold=-1)

In [ ]:
def get_swap_circuit(index: int):
    if index < 0:
        return QuantumCircuit(num_physical_qubits)
    from_layout = Layout({i: compiled_circuits[index].qubits[edge_maps[index][i]] for i in range(num_physical_qubits)})
    for instruction in compiled_circuits[index].data:
        if instruction.operation.name == 'swap':
            qubits_str = str(instruction.qubits)
            matches = re.findall('index=([0-9]+)', qubits_str)
            if len(matches) == 2:
                from_layout.swap(compiled_circuits[0].qubits[int(matches[0])], compiled_circuits[0].qubits[int(matches[1])])
            else:
                raise Exception('Did not find 2 swap indices')
    to_layout = Layout({i: compiled_circuits[index+1].qubits[edge_maps[index+1][i]] for i in range(num_physical_qubits)})
    transformation_pass = LayoutTransformation(extended_swap_strat._coupling_map, from_layout, to_layout)
    swap_qc = QuantumCircuit(num_physical_qubits)
    swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
    return swap_qc

In [ ]:
qaoa_circuit = QuantumCircuit(num_qubits, num_qubits)

mixer_layer = QuantumCircuit(num_qubits)
beta = Parameter("β")
mixer_layer.rx(2 * beta, range(num_qubits))

gammas = ParameterVector("γ", args.reps)
betas = ParameterVector("β", args.reps)

for i in range(num_qubits):
    qaoa_circuit.h(i)
qaoa_circuit.barrier()

for layer in range(0, args.reps):
    swap_circuit = get_swap_circuit(layer-1)
    qaoa_circuit.compose(swap_circuit, range(num_qubits), inplace=True)

    bind_dict = {compiled_circuits[layer].parameters[0]: gammas[layer]}
    bound_cost_layer = compiled_circuits[layer].assign_parameters(bind_dict)
    qaoa_circuit.compose(bound_cost_layer, range(num_qubits), inplace=True)
   
    bind_dict = {mixer_layer.parameters[0]: betas[layer]}
    bound_mixer_layer = mixer_layer.assign_parameters(bind_dict)
    qaoa_circuit.compose(bound_mixer_layer, range(num_qubits), inplace=True)

layer = args.reps - 1
final_layout = Layout({i: compiled_circuits[layer].qubits[edge_maps[layer][i]] for i in range(num_qubits)})
for instruction in compiled_circuits[layer].data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            final_layout.swap(compiled_circuits[0].qubits[int(matches[0])], compiled_circuits[0].qubits[int(matches[1])])
        else:
            raise Exception('Did not find 2 swap indices')

for cidx, qidx in (
    final_layout.get_physical_bits().items()
):
    qaoa_circuit.measure(qidx, cidx)

In [ ]:
qaoa_circuit.draw(fold=-1)

In [ ]:
# Now transpile to basis gates
generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, basis_gates=basis_gates)
init  = generic_pm.init
init.remove(3)
generic_pm.init = init
generic_pm.layout = None
t_qaoa_circ = generic_pm.run(qaoa_circuit)

print_circuit_info(t_qaoa_circ, 'QAOA circuit')
print(t_qaoa_circ.count_ops())


In [ ]:
b_qaoa_circ = t_qaoa_circ.assign_parameters({param: 0.25 for param in t_qaoa_circ.parameters})

In [ ]:
results = backend.run(b_qaoa_circ).result()

In [ ]:
results